# Moon Rover Optimal Control Testing

Test MPC cost function over the **entire** horizon.

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import jax.numpy as jnp
import numpy as np
import pandas as pd
import scipy.optimize as sci_opt
import matplotlib.pyplot as plt

import exp_mpc.stewart_min.robo as robo
import exp_mpc.stewart_min.vest as vest
import exp_mpc.stewart_min.quartic_cost as quartic_cost
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.viz as viz

In [ ]:
def load_specific_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    df = pd.read_csv(file_path)

    ks = df.keys()

    ts = np.array(df[ks[0]])
    diff = np.abs(np.diff(ts))
    avg_diff = np.mean(diff)
    std_diff = np.std(diff)
    if std_diff > 0.05:
        bad_indices = np.where(diff > avg_diff + std_diff)[0] + 1 # off by one
        start_index = bad_indices[-2] + 5 * 200
        end_index = bad_indices[-1] - 1
    else:
        start_index = 0
        end_index = ts.size - 1

    df = df[start_index: end_index + 1]

    acc_ref = jnp.transpose(jnp.array([df[k] for k in ks[1:4]]))
    omega_ref = jnp.transpose(jnp.array([df[k] for k in ks[4:]]))
    return acc_ref, omega_ref

In [ ]:
# load data

# file_path = "/Users/jozbee/work/eng/comp/data/sms_00_sms_drive.csv"
file_path = "/Users/jozbee/work/eng/comp/data/sms_specific-forces-standard-road-v2.csv"
acc_ref, omega_ref = load_specific_sms_references(file_path)

max_size = 100 * 200
acc_ref = acc_ref[:max_size]
omega_ref = omega_ref[:max_size]

assert acc_ref.shape[0] == omega_ref.shape[0]
assert acc_ref.shape[1] == 3
assert omega_ref.shape[1] == 3

In [ ]:
# cost setup

params = robo.RoboParams(dt_mpc=robo.RoboParams.dt)
geom = robo.RoboGeom()

weights = opt.ExpWeights(
    acc=jnp.array([1e2, 1e2, 1e0]),
    omega=jnp.array([2e2, 2e2, 2e2]),
    alpha_acc=jnp.array([0.0]),
    alpha_omega=jnp.array([0.0]),
)

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]
euler_margins = [0.2 / 3.0, 0.1 / 3.0]
euler_sizes = [2**0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    low=params.leg_min,
    high=params.leg_max,
    center=geom.lengths_home,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-params.max_leg_vel,
    high=params.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-params.joint_max_angle,
    high=params.joint_max_angle,
)
roll_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-params.max_roll,
    high=params.max_roll,
)
pitch_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-params.max_pitch,
    high=params.max_pitch,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-params.max_rotary_yaw,
    high=params.max_rotary_yaw,
)
yaw_dot_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-params.max_rotary_vel,
    high=params.max_rotary_vel,
)
cost_terms = opt.CostTerms(
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    roll_cost=roll_cost,
    pitch_cost=pitch_cost,
    yaw_cost=yaw_cost,
    yaw_dot_cost=yaw_dot_cost,
)

In [ ]:
# opt

vspec_acc = vest.acc_vspec0
vspec_omega = vest.omega_vspec0

acc_num = vspec_acc.E0.shape[0]
omega_num = vspec_omega.E0.shape[0]
v_num = 3 * (acc_num + omega_num)
gr = (2 * acc_num, 3 * acc_num)
v0_earth = jnp.zeros(v_num)
v0_moon = jnp.zeros(v_num)

v0_earth = v0_earth.at[gr[0]: gr[1]].set(vspec_acc.v0_earth)
v0_moon = v0_moon.at[gr[0]: gr[1]].set(vspec_acc.v0_moon)

def sci_fun(u):
    res = opt.cost_and_grad_flat_jax(
        weights=weights,
        cost_terms=cost_terms,
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=jnp.zeros(12),
        vstate0_irl=v0_earth,
        vstate0_sim=v0_moon,
        control0=jnp.zeros(6),
        control_flat=u,
        use_rotary=True,
        use_terminal=False,
    )
    return [np.array(arr) for arr in res]

res = sci_opt.minimize(
    fun=sci_fun,
    x0=np.zeros(acc_ref.shape[0] * 6),
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 100000,
        "maxfun": 100000,
        "ftol": 1e-10,
        "gtol": 1e-10,
    },
)

In [ ]:
res

In [ ]:
np.mean(np.abs(res.jac)), np.std(np.abs(res.jac))

In [ ]:
u = utils.Control.from_flat(jnp.array(res.x))
x = utils.get_rstate(dt=params.dt, control=u, rstate0=jnp.zeros(12))
vstate_irl = utils.get_vstate_irl(
    vspec_acc,
    vspec_omega,
    x,
    u,
    jnp.zeros(6),
    v0_earth,
)
vstate_sim = utils.get_vstate(
    vspec_acc, vspec_omega, acc_ref, omega_ref, v0_moon
)

In [ ]:
trajectory = []
for i in range(acc_ref.shape[0]):
    sol = utils.TableSol(
        x=utils.RState(x.state[i]),
        u=utils.Control(u.control[i]),
        vstate_irl=utils.VState(vstate_irl.x_state[i], vstate_irl.y_state[i]),
        vstate_sim=utils.VState(vstate_sim.x_state[i], vstate_sim.y_state[i]),
        stats=utils.TableStats(jnp.array(0.0), jnp.array(0), jnp.array(0.0)),
    )
    trajectory.append(sol)

In [ ]:
plt.close("all")

In [ ]:
references = {
    "xyz-acceleration": jnp.array(acc_ref),
    "angular-velocity": jnp.array(omega_ref),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=trajectory, references=references, robo_params=params)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=trajectory, robo_params=params)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=trajectory, robo_params=params)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=trajectory, robo_params=params, robo_geom=geom)